# Appendix E - Operational Decision Report

This notebook is part of the reproducibility package for the MSc thesis **Probabilistic Machine Learning Models and Decision Optimization for Hepatocellular Carcinoma Diagnosis**.

> Repository note: paths are resolved from either the repository root or the `notebooks/` directory. Generated figures are written under `outputs/`.


# End-to-End Pipeline + Operational Decision Report
# Treat rate + Confusion Matrix per Scenario

In [1]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.utils import resample

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    roc_curve, precision_recall_curve,
    confusion_matrix
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

from xgboost import XGBClassifier
# Resolve repository root for reproducible relative paths
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists() and (REPO_ROOT.parent / "data").exists():
    REPO_ROOT = REPO_ROOT.parent


In [2]:
# Basic playability/display settings
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.precision", 4)

In [3]:
TARGET = "liver_cancer"
df = pd.read_csv(REPO_ROOT / "data" / "synthetic_liver_cancer_dataset.csv")

In [4]:
y = df[TARGET].astype(int).values
X = df.drop(columns=[TARGET])

In [5]:
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

In [6]:
display(pd.DataFrame({
    "numeric_cols_count": [len(num_cols)],
    "categorical_cols_count": [len(cat_cols)],
    "class_0_ratio": [(y == 0).mean()],
    "class_1_ratio": [(y == 1).mean()],
}))

,numeric_cols_count,categorical_cols_count,class_0_ratio,class_1_ratio
0,9,4,0.7822,0.2178


In [7]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=RANDOM_STATE
)

In [9]:
print("TRAIN size:", X_train.shape, "VAL size:", X_val.shape, "TEST size:", X_test.shape)

TRAIN size: (3000, 13) VAL size: (1000, 13) TEST size: (1000, 13)


In [10]:
numeric_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [11]:
categorical_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [12]:
preprocessor = ColumnTransformer([
    ("num", numeric_tf, num_cols),
    ("cat", categorical_tf, cat_cols)
])

In [13]:
log_reg = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    solver="lbfgs"
)

In [14]:
rf = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

In [15]:
pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=RANDOM_STATE,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    n_jobs=-1,
    scale_pos_weight=pos_weight
)

In [16]:
pipe_lr = Pipeline([("prep", preprocessor), ("clf", log_reg)])

In [17]:
pipe_rf = Pipeline([("prep", preprocessor), ("clf", rf)])

In [18]:
pipe_xgb = Pipeline([("prep", preprocessor), ("clf", xgb)])

In [19]:
param_grid_lr = {
    "clf__C": [0.1, 1.0, 3.0],
    "clf__solver": ["lbfgs", "liblinear"]
}

In [20]:
param_grid_rf = {
    "clf__n_estimators": [300, 600],
    "clf__max_depth": [None, 8],
}

In [21]:
param_grid_xgb = {
    "clf__n_estimators": [300, 600],
    "clf__max_depth": [3, 5],
    "clf__learning_rate": [0.03, 0.07],
}

In [22]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [23]:
def tune(pipe, grid, name):
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=0
    )
    gs.fit(X_train, y_train)
    return {"model": name, "cv_best_roc_auc": gs.best_score_, "best_params": gs.best_params_}, gs.best_estimator_

In [24]:
res_lr, best_lr = tune(pipe_lr, param_grid_lr, "LogisticRegression")

In [25]:
res_rf, best_rf = tune(pipe_rf, param_grid_rf, "RandomForest")

In [26]:
res_xgb, best_xgb = tune(pipe_xgb, param_grid_xgb, "XGBoost")

In [27]:
tuning_tbl = pd.DataFrame([res_lr, res_rf, res_xgb])
display(tuning_tbl)

,model,cv_best_roc_auc,best_params
0,LogisticRegression,0.9657,"{'clf__C': 1.0, 'clf__solver': 'lbfgs'}"
1,RandomForest,0.9861,"{'clf__max_depth': 8, 'clf__n_estimators': 600}"
2,XGBoost,0.9993,"{'clf__learning_rate': 0.07, 'clf__max_depth':..."


In [28]:
cv_cal = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [29]:
def pick_calibration(model, X_tr, y_tr, X_val, y_val, cv_cal):
    methods = {}
    scores = {}
    for m in ["isotonic", "sigmoid"]:
        cal = CalibratedClassifierCV(model, method=m, cv=cv_cal)
        cal.fit(X_tr, y_tr)
        proba_val = cal.predict_proba(X_val)[:, 1]
        scores[m] = brier_score_loss(y_val, proba_val)
        methods[m] = cal
    best_m = min(scores, key=scores.get)  # smaller Brier = better calibration
    return methods[best_m], best_m, scores

In [30]:
cal_lr, m_lr, s_lr = pick_calibration(best_lr, X_train, y_train, X_val, y_val, cv_cal)

In [31]:
cal_rf, m_rf, s_rf = pick_calibration(best_rf, X_train, y_train, X_val, y_val, cv_cal)

In [32]:
cal_xgb, m_xgb, s_xgb = pick_calibration(best_xgb, X_train, y_train, X_val, y_val, cv_cal)

In [33]:
calibration_tbl = pd.DataFrame([
    {"model": "LogisticRegression", "picked": m_lr, "val_brier": s_lr[m_lr]},
    {"model": "RandomForest", "picked": m_rf, "val_brier": s_rf[m_rf]},
    {"model": "XGBoost", "picked": m_xgb, "val_brier": s_xgb[m_xgb]},
])
display(calibration_tbl)

,model,picked,val_brier
0,LogisticRegression,sigmoid,0.0587
1,RandomForest,isotonic,0.0276
2,XGBoost,sigmoid,0.0010


In [34]:
X_dev_final = pd.concat([X_train, X_val], axis=0)
y_dev_final = np.concatenate([y_train, y_val], axis=0)

In [35]:
cal_lr.fit(X_dev_final, y_dev_final)

,estimator,Pipeline(step..._iter=5000))])
,method,'sigmoid'
,cv,StratifiedKFo... shuffle=True)
,n_jobs,None
,ensemble,'auto'
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False


In [36]:
cal_rf.fit(X_dev_final, y_dev_final)

,estimator,Pipeline(step...m_state=42))])
,method,'isotonic'
,cv,StratifiedKFo... shuffle=True)
,n_jobs,None
,ensemble,'auto'
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False


In [37]:
cal_xgb.fit(X_dev_final, y_dev_final)

,estimator,"Pipeline(step...=None, ...))])"
,method,'sigmoid'
,cv,StratifiedKFo... shuffle=True)
,n_jobs,None
,ensemble,'auto'
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False


In [38]:
def bootstrap_ci(y_true, proba, metric_fn, n_boot=2000, alpha=0.05, rng=RANDOM_STATE):
    rng = np.random.RandomState(rng)
    n = len(y_true)
    stats = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        stats.append(metric_fn(y_true[idx], proba[idx]))
    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(np.mean(stats)), (float(lo), float(hi))

In [39]:
def evaluate(model, name):
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    ap = average_precision_score(y_test, proba)
    brier = brier_score_loss(y_test, proba)
    _, auc_ci = bootstrap_ci(y_test, proba, roc_auc_score)
    _, ap_ci = bootstrap_ci(y_test, proba, average_precision_score)
    return {"model": name, "auc": auc, "auc_ci": auc_ci, "ap": ap, "ap_ci": ap_ci, "brier": brier}, proba

In [40]:
m_lr, p_lr = evaluate(cal_lr, "LR")

In [41]:
m_rf, p_rf = evaluate(cal_rf, "RF")

In [42]:
m_xgb, p_xgb = evaluate(cal_xgb, "XGB")

In [43]:
summary = pd.DataFrame([m_lr, m_rf, m_xgb])[["model", "auc", "auc_ci", "ap", "ap_ci", "brier"]]
summary = summary.sort_values("auc", ascending=False)
display(summary)

,model,auc,auc_ci,ap,ap_ci,brier
2,XGB,1.0000,"(0.9999445656084974, 1.0)",1.0000,"(0.9997987533980084, 1.0)",0.0013
1,RF,0.9926,"(0.9883858749044776, 0.9962251293922882)",0.9788,"(0.9677472480203282, 0.98830171677598)",0.0273
0,LR,0.9671,"(0.9571400547805493, 0.9762852290563879)",0.8961,"(0.8618734677683105, 0.9267034814861587)",0.0593


In [44]:
model_selection_tbl = tuning_tbl.merge(
    calibration_tbl[["model", "val_brier"]],
    on="model",
    how="left"
)
display(model_selection_tbl)

,model,cv_best_roc_auc,best_params,val_brier
0,LogisticRegression,0.9657,"{'clf__C': 1.0, 'clf__solver': 'lbfgs'}",0.0587
1,RandomForest,0.9861,"{'clf__max_depth': 8, 'clf__n_estimators': 600}",0.0276
2,XGBoost,0.9993,"{'clf__learning_rate': 0.07, 'clf__max_depth':...",0.0010


In [45]:
winner_row = model_selection_tbl.sort_values(
    ["cv_best_roc_auc", "val_brier"],
    ascending=[False, True]
).iloc[0]
winner_cv = winner_row["model"]
print("Best model (TRAIN CV ROC-AUC + VAL Brier tie-break):", winner_cv)

Best model (TRAIN CV ROC-AUC + VAL Brier tie-break): XGBoost


In [54]:
# Operational Decision Report
# Treat rate + Confusion Matrix per scenario (Part II bridge)
# "Treat" here is interpreted as: "refer to additional / more specialized diagnostic work-up"
# (e.g., extra imaging, biopsy, labs) and NOT necessarily immediate definitive treatment.

In [55]:
# Winner model probabilities on TEST (already computed in the pipeline)
proba_map = {
    "LogisticRegression": p_lr,
    "RandomForest": p_rf,
    "XGBoost": p_xgb
}
proba_test = proba_map[winner_cv]

In [56]:
# Payoff scenarios (use the exact values from your Part II tables)
scenarios = {
    # Clinical (QALYs)
    "S1_Clinical_Balanced":     {"U11": -0.060,   "U10": -0.130,   "U01": -0.039,   "U00": 0.000},
    "S2_Clinical_Conservative": {"U11": -0.060,   "U10": -0.650,   "U01": -0.039,   "U00": 0.000},
    "S3_Clinical_Aggressive":   {"U11": -0.060,   "U10": -0.130,   "U01": -0.195,   "U00": 0.000},

    # Economic (USD)
    "S4_Econ_Balanced":         {"U11": -62775.0, "U10": -92133.0, "U01": -1171.0,  "U00": 0.0},
    "S5_Econ_Conservative":     {"U11": -177658.0,"U10": -260737.0,"U01": -1171.0,  "U00": 0.0},
    "S6_Econ_Aggressive":       {"U11": -62775.0, "U10": -92133.0, "U01": -23420.0, "U00": 0.0},
}

In [57]:
def p_star_from_payoffs(U11, U10, U01, U00):
    # Pauker–Kassirer: p* = LFP / (GTP + LFP), where LFP=U00-U01 and GTP=U11-U10
    LFP = U00 - U01
    GTP = U11 - U10
    denom = GTP + LFP
    if denom <= 0:
        raise ValueError(f"Invalid payoffs: GTP+LFP must be > 0, got {denom:.6f}")
    return float(LFP / denom)

In [58]:
def policy_report_row(y_true, proba, thr, U):
    """
    Apply policy: Escalate if proba >= thr.
    Return one row with Escalate rate, confusion matrix, operational metrics,
    and realized expected utility on TEST given the same Uij payoffs.
    """
    y_pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    n = len(y_true)
    escalate_n = int(y_pred.sum())
    escalate_rate = escalate_n / n

    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    ppv  = tp / (tp + fp) if (tp + fp) else np.nan
    npv  = tn / (tn + fn) if (tn + fn) else np.nan

    # Realized total utility on TEST under this policy (counts-weighted)
    EU_total = (tp * U["U11"]) + (fn * U["U10"]) + (fp * U["U01"]) + (tn * U["U00"])
    EU_per_patient = EU_total / n

    return {
        "p*": float(thr),
        "Escalate_n": escalate_n,
        "Escalate_%": float(escalate_rate),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Sensitivity": float(sens),
        "Specificity": float(spec),
        "PPV": float(ppv),
        "NPV": float(npv),
        "EU_per_patient": float(EU_per_patient),
        "EU_total": float(EU_total),  # optional but useful in Appendix
    }

In [59]:
# Single-line summary (keep or remove)
print(f"Operational policy table on TEST | winner={winner_cv} | n={len(y_test)} | prevalence(y=1)={(y_test==1).mean():.4f}")

Operational policy table on TEST | winner=XGBoost | n=1000 | prevalence(y=1)=0.2180


In [60]:
rows = []
for scen_name, U in scenarios.items():
    thr = p_star_from_payoffs(U["U11"], U["U10"], U["U01"], U["U00"])
    row = policy_report_row(y_test, proba_test, thr, U)
    row["Scenario"] = scen_name
    rows.append(row)

In [61]:
op_tbl = pd.DataFrame(rows)[[
    "Scenario", "p*", "Escalate_n", "Escalate_%",
    "TP", "FP", "FN", "TN",
    "Sensitivity", "Specificity", "PPV", "NPV",
    "EU_per_patient", "EU_total"
]].sort_values("p*")

display(op_tbl)

,Scenario,p*,Escalate_n,Escalate_%,TP,FP,FN,TN,Sensitivity,Specificity,PPV,NPV,EU_per_patient,EU_total
4,S5_Econ_Conservative,0.0139,238,0.238,218,20,0,762,1.0000,0.9744,0.9160,1.0000,-38752.8640,-3.8753e+07
3,S4_Econ_Balanced,0.0384,226,0.226,218,8,0,774,1.0000,0.9898,0.9646,1.0000,-13694.3180,-1.3694e+07
1,S2_Clinical_Conservative,0.0620,224,0.224,218,6,0,776,1.0000,0.9923,0.9732,1.0000,-0.0133,-1.3314e+01
0,S1_Clinical_Balanced,0.3578,220,0.220,218,2,0,780,1.0000,0.9974,0.9909,1.0000,-0.0132,-1.3158e+01
5,S6_Econ_Aggressive,0.4437,220,0.220,218,2,0,780,1.0000,0.9974,0.9909,1.0000,-13731.7900,-1.3732e+07
2,S3_Clinical_Aggressive,0.7358,218,0.218,217,1,1,781,0.9954,0.9987,0.9954,0.9987,-0.0133,-1.3345e+01
